In [ ]:
!pip install qiskit
!pip install qiskit-ibm-runtime
!pip install qiskit_aer
!pip install pylatexenc

Creamos nuestro primer circuito: tendrá un qubit, una puerta H y una medición.

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(1,1)
qc.h(0)
qc.measure(0,0)

qc.draw("mpl")

Utilizamos un simulador para ejecutar el circuito. Utilizamos 10 shots (ejecuciones) y una semilla (1234) para que los resultados sean reproducibles.

In [ ]:
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_aer import AerSimulator


sampler = Sampler(AerSimulator(seed_simulator = 1234))
job = sampler.run([qc], shots = 10)
results = job.result()
d = results[0].data.c # Acceso a los valores de las mediciones
print("Número de ejercuciones:", d.num_shots)
print("Estadísticas de las ejecuciones:", d.get_counts())
print("Resultados de las medidas:", d.array)
print("Resultados de las medidas como cadenas de texto:", d.get_bitstrings())


Ahora, vamos a calcular de forma exacta las amplitudes del vector de estado final. Esto solo se puede hacer un simulador, no en un ordenador cuántico real.

Es necesario eliminar las mediciones finales y usar la función *Statevector*.

In [ ]:
qc.remove_final_measurements()
from qiskit.quantum_info import Statevector
statevector = Statevector(qc)
print("Amplitudes del vector de estado:", statevector)

Ahora, vamos a calcular la matriz del circuito. De nuevo, esto solo se puede hacer en un simulador. Usaremos la función *Operator*.

In [ ]:
from qiskit.quantum_info import Operator
result = Operator(qc).data
print("Matriz del circuito:", result)

##BONUS: EJECUCIÓN EN UN ORDENADOR CUÁNTICO REAL

Para ejecutar en un ordenador cuántico real, necesitamos acceder a los servicios de IBM Quantum. Debes copiar tu token de https://quantum.ibm.com/ y pegarlo dentro de las comillas de la asignación de la variable *mytoken*.

In [ ]:
mytoken = ""

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService(channel="ibm_cloud", token=mytoken)

Ahora, necesitamos hacer dos cosas más:

* Seleccionar un ordenador cuántico. Elegiremos el menos ocupado con el método *least_busy*.
* Transpilar el circuito para poder ejecutarlo en el ordenador que hemos elegido

In [ ]:
backend = service.least_busy(simulator=False, operational=True)
print("El ordenador seleccionado ha sido:", backend)

qc2 = QuantumCircuit(1,1)
qc2.h(0)
qc2.measure(0,0)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc2)

isa_circuit.draw('mpl', idle_wires=False)

Ahora, ya podemos ejecutarlo como hacíamos en el simulador.

In [ ]:
sampler = Sampler(backend)
job = sampler.run([isa_circuit], shots = 10)
results = job.result()
d = results[0].data.c # Acceso a los valores de las mediciones
print("Número de ejercuciones:", d.num_shots)
print("Estadísticas de las ejecuciones:", d.get_counts())
print("Resultados de las medidas:", d.array)
print("Resultados de las medidas como cadenas de texto:", d.get_bitstrings())